In [ ]:
import yfinance as yf
import pandas as pd, numpy as np
import xgboost as xgb, lightgbm as lgb
import ta
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from sklearn.metrics import classification_report, recall_score, precision_score, f1_score, confusion_matrix, accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.utils.class_weight import compute_sample_weight
warnings.filterwarnings("ignore")

In [ ]:
def evaluate_model(y_true, y_pred, model_name):
    precision = precision_score(y_true, y_pred, average="weighted")
    recall = recall_score(y_true, y_pred, average="weighted")
    f1 = f1_score(y_true, y_pred, average="weighted")
    cm = confusion_matrix(y_true, y_pred)
    return {
        "model": model_name,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "confusion_matrix": cm,
    }

In [ ]:
def train_base_model(X_train, y_train, X_valid=None, y_valid=None, model_name="LOG"):
    sc = None
    train_weights = compute_sample_weight('balanced', y_train)
    val_weights = compute_sample_weight('balanced', y_valid)
    if model_name == "LOG":
        sc = StandardScaler()
        model = LogisticRegression(class_weight="balanced", max_iter=1000)
        X_train_scaled = sc.fit_transform(X_train)
        model.fit(X_train_scaled, y_train)
    elif model_name == "XGB":
        model = xgb.XGBClassifier(n_estimators=1000, early_stopping_rounds=50, objective="multi:softprob", num_class=3,learning_rate=0.05,max_depth=6,subsample=0.8,colsample_bytree=0.8)
        model.fit(X_train,y_train,eval_set=[(X_valid, y_valid)],verbose=False,sample_weight=train_weights, sample_weight_eval_set=[val_weights])
    elif model_name == 'LGB':
        model = lgb.LGBMClassifier(n_estimators=1000, random_state=1, objective="multiclass", num_class=3, class_weight="balanced")
        model.fit(X_train,y_train,eval_set=[(X_valid, y_valid)], callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)])
    return model, sc

In [ ]:
TICKER = 'BTC-USD'
PERIOD = "3y"
Ticker = yf.Ticker(ticker=f'{TICKER}')
df = Ticker.history(PERIOD)
start = df.index.min()
end = df.index.max()
sample_size = len(df)
df.drop(columns=['Dividends','Stock Splits'], inplace=True)
print(f'✅ Downloaded {TICKER} data from {start} to {end}, total bars: {sample_size}')

In [ ]:
df["SMA9"] = df["Close"].rolling(window=9).mean()
df["SMA25"] = df["Close"].rolling(window=25).mean()
df["Close_SMA9_ratio"] = (df["Close"] - df["SMA9"]) / df["SMA9"]
df["Close_SMA25_ratio"] = (df["Close"] - df["SMA25"]) / df["SMA25"]
rsi = ta.momentum.RSIIndicator(close=df["Close"])
df["RSI"] = rsi.rsi()
macd = ta.trend.MACD(close=df["Close"])
df["MACD"] = macd.macd()
df["Signal_Line"] = macd.macd_signal()
df["MACD_Diff"] = ta.trend.macd_diff(df["Close"])
df["Volatility9"] = df["Close"].rolling(window=9).std()
df["Volatility25"] = df["Close"].rolling(window=25).std()
df["Support_Level_short"] = df["Close"].rolling(window=25).min()
df["Resistance_Level_short"] = df["Close"].rolling(window=25).max()
for lag in [1, 2, 3, 5, 7, 9, 25]:
    df[f"Price_Change_{lag}d"] = df["Close"].pct_change(lag)
    df[f"Vol_Change_{lag}d"] = df["Volume"].pct_change(lag)
df["Price_Change_5d_later"] = df["Price_Change_5d"].shift(-5)
df.loc[df["Price_Change_5d_later"] >= 0.04, "Target"] = 0  # Buy
df.loc[df["Price_Change_5d_later"] <= -0.04, "Target"] = 2  # Sell
df.loc[df["Target"].isna(), "Target"] = 1  # Hold
df.dropna(inplace=True)
print(f"✅ Technical indicators calculated...")
df.dropna(inplace=True)
print(f'✅ Data cleaned...')

In [ ]:
target_labels = {0:'Buy', 1:'Hold', 2:'Sell'}
class_weights = df['Target'].value_counts(normalize=True).sort_index().to_dict()

print(f'✅ Target variable added, class weights:')
for k, v in class_weights.items():
    print(f"{target_labels[k]} {v:.0%}")

In [ ]:
features = ['Volume', 'Close_SMA9_ratio', 'Close_SMA25_ratio', 'RSI', 'Volatility9', 'Volatility25', 'Price_Change_1d', 'Vol_Change_1d', 'Price_Change_2d', 'Vol_Change_2d',
       'Price_Change_3d', 'Vol_Change_3d', 'Price_Change_5d', 'Vol_Change_5d', 'Price_Change_7d', 'Vol_Change_7d', 'Price_Change_9d', 'Vol_Change_9d', 'Price_Change_25d', 'Vol_Change_25d']

In [ ]:
train_end_date = "2025-06-30"

train_data = df[:train_end_date]
test_data = df[train_end_date:]
X_train, y_train = train_data[features], train_data["Target"]
X_test, y_test = test_data[features], test_data["Target"]
train_size = len(X_train)
test_size = len(X_test)
print(f"✅ Data split done. Train: {train_size}, Test: {test_size}")

In [ ]:
splits = 5
tscv = TimeSeriesSplit(n_splits=splits)
metrics = []
for train_idx, valid_idx in tscv.split(X_train, y_train):
    x_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
    x_va, y_va = X_train.iloc[valid_idx], y_train.iloc[valid_idx]
    
    xgb_cl, _ = train_base_model(x_tr, y_tr, x_va, y_va, model_name='XGB')
    y_pred = xgb_cl.predict(x_va)
    results = evaluate_model(y_va, y_pred, 'XGB')
    metrics.append(results)
print('Cross Validation Results:\n', pd.DataFrame(metrics).to_string(max_colwidth=50))

In [ ]:
y_test_pred = xgb_cl.predict(X_test)
test_results = evaluate_model(y_test, y_test_pred, 'XGB')
print('Test Results:\n', pd.DataFrame([test_results]).to_string(max_colwidth=50))

In [ ]:
print('Test Prediction Class proportions:')
print(pd.Series(y_test_pred).value_counts(normalize=True).sort_index())

In [ ]:
df_test = test_data.copy()
df_test["pred"] = y_test_pred

signal_map = {0: 1, 1: 0, 2: -1}
df_test["signal"] = df_test["pred"].map(signal_map)

df_test["ret"] = df_test["Close"].pct_change()
df_test["strategy_ret"] = df_test["signal"].shift(1) * df_test["ret"]

# Performance
cum_return = (1 + df_test["strategy_ret"]).cumprod()
sharpe = df_test["strategy_ret"].mean() / df_test["strategy_ret"].std() * np.sqrt(252)

print("Sharpe:", sharpe)